# ChebAr: does `rescale_filter` change the answer?

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

`ChebAr(..., rescale_filter=True)` normalizes the input of every propagator call
inside the Chebyshev recurrence and scales the output back. `P` is linear, so
that is exact on paper — but it changes which tolerance the adaptive integrator
is actually enforcing, so it has to be checked numerically, not argued.

**Why it should be faster.** The Krylov vectors are unit-norm over
`dim = N**2` entries, so a typical entry is `~1/sqrt(dim)` and
`rtol*|y| << atol`: the integrator runs in the **atol-dominated** regime.
The recurrence `t_2 = 2*w*t_1 - t_0` then grows `|t_k|` geometrically, entries
reach `O(1)`, `rtol*|y|` takes over and the same `mesolve` is suddenly held to a
much tighter relative accuracy — which is the observed "first call of a filter
fast, the remaining `cheb_degree - 1` slow" pattern. Rescaling puts every call
back in the first regime, the same one `first_estimation` and `ArnoldiLindblad`
already live in.

**Why it might cost accuracy.** That regime is looser. The question this
notebook answers is whether the loss shows up in `mu`, `rate_bf` and `res_rel`
at a level that matters.

Both variants run **in one job, on one GPU, from the same `x0` and the same
ellipse** — `first_estimation` is done once and its `ritz_vals` are fed to both
`setup_chebyshev` calls, so the Chebyshev filters are identical polynomials and
the rescaling is the only difference between the two runs.

Structure, kernel and auth are as in `notebooks/solvers/cheb_ar.ipynb`; the cell below
is copied from it verbatim. Expect roughly **4–6 minutes**.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

In [ ]:
def run_rescale_compare(
    n_a=25,
    n_b=11,
    alpha_sq=4.25,
    m_arnoldi_0=60,
    m_arnoldi=120,
    margin=1e-2,
    seed=0,
    cheb_degree=6,
):
    """Run the ChebAr filtered Arnoldi twice, with and without `rescale_filter`.

    Everything upstream of the filter is shared: one model build, one `x0`, one
    `first_estimation`, one set of `ritz_vals`. Both solvers are then given that
    same spectrum estimate, so they build the same ellipse and the same
    polynomial, and the only difference left is how the propagator is called
    inside the recurrence.

    Returns plain numpy/python objects only (the client has neither jax nor
    dynamiqs); `Q` never crosses the wire.
    """
    import time

    import numpy as np

    from floquet_lindblad import ChebAr
    from floquet_lindblad.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    def time_apply(fn, x, repeats=3):
        """Seconds per call of `fn` on `x`, jit warm-up excluded.

        `block_until_ready` is mandatory: jax dispatch is asynchronous, so
        without it this times the launch, not the solve.
        """
        fn(x).block_until_ready()
        t = time.time()
        for _ in range(repeats):
            y = fn(x)
        y.block_until_ready()
        return (time.time() - t) / repeats

    dims = (n_a, n_b)

    # --- shared: model, seed, spectrum estimate --------------------------- #
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction(n_a=n_a, n_b=n_b, alpha_sq=alpha_sq)

    def make_solver(rescale):
        return ChebAr(
            H_I,
            jump_ops_I,
            T_block,
            jump_ops_LdL=jump_ops_LdL_I,
            dims=dims,
            output_phase=output_phase,
            cheb_degree=cheb_degree,
            rescale_filter=rescale,
        )

    solver_ref = make_solver(True)
    x0 = solver_ref.make_x0(seed=seed)

    t0 = time.time()
    _, _, ritz_vals = solver_ref.first_estimation(x0, m_arnoldi=m_arnoldi_0)
    t_first_estimation = time.time() - t0

    # --- the regime effect itself, before any Arnoldi --------------------- #
    # The bare propagator, on a unit-norm vector (what the Krylov vectors look
    # like) and on the same vector scaled up by 1e3 (what the recurrence hands
    # it a couple of steps in). Same operator, same work on paper -- the gap is
    # the adaptive integrator switching from atol- to rtol-dominated, and it is
    # the whole reason `rescale_filter` exists.
    propagate = solver_ref.propagate_block_projected
    t_apply_unit = time_apply(propagate, x0)
    t_apply_big = time_apply(propagate, 1e3 * x0)

    # --- the two filtered runs -------------------------------------------- #
    out = {}
    for key, rescale in (("rescaled", True), ("plain", False)):
        solver = solver_ref if rescale else make_solver(False)
        # Same ritz_vals -> same ellipse -> same polynomial for both.
        ellipse = solver.setup_chebyshev(ritz_vals, margin=margin)

        t_filter = time_apply(solver.chebyshev_filter, x0)

        t0 = time.time()
        Q, H, mu_list = solver.arnoldi_hessenberg(
            x0, solver.chebyshev_filter, m_arnoldi, warm_start=False
        )
        t_arnoldi = time.time() - t0

        rate_bf = solver.rate_from_mu(mu_list[-1])
        x_ritz, _ = solver.ritz_vector(Q, H, m_arnoldi)
        res = solver.residual_check(x_ritz)
        del Q, x_ritz  # hundreds of MB; must not survive into the next variant

        out[key] = {
            "rescale_filter": rescale,
            "mu_list": np.asarray(mu_list),
            "mu": py(mu_list[-1]),
            "rate_bf": py(rate_bf),
            "mu_RQ": py(res["mu_RQ"]),
            "rate_RQ": py(res["rate_RQ"]),
            "res_rel": py(res["res_rel"]),
            "t_filter": t_filter,
            "t_arnoldi": t_arnoldi,
            "ellipse": {k: py(v) for k, v in ellipse.items() if np.ndim(v) == 0},
        }
        if not rescale:
            del solver

    import jax

    return {
        "variants": out,
        "T_block": py(T_block),
        "t_first_estimation": t_first_estimation,
        "t_apply_unit": t_apply_unit,
        "t_apply_big": t_apply_big,
        "params": {k: py(v) for k, v in params.items()},
        "settings": {
            "n_a": n_a,
            "n_b": n_b,
            "alpha_sq": alpha_sq,
            "m_arnoldi_0": m_arnoldi_0,
            "m_arnoldi": m_arnoldi,
            "margin": margin,
            "seed": seed,
            "cheb_degree": cheb_degree,
        },
        "device": str(jax.devices()[0]),
    }

In [ ]:
import time

t0 = time.time()
result = acr.run(
    run_rescale_compare,
    num_gpus=1,
    num_cpus=8,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s")
print("device :", result["device"])

In [ ]:
import numpy as np

resc = result["variants"]["rescaled"]
plain = result["variants"]["plain"]

# The two ellipses must be identical -- both solvers were handed the same
# ritz_vals. If they are not, the comparison is not controlled and nothing
# below means anything.
same_ellipse = all(
    np.isclose(resc["ellipse"][k], plain["ellipse"][k])
    for k in resc["ellipse"]
    if k in plain["ellipse"]
)
print("same ellipse for both variants:", same_ellipse)

print("\n--- the regime effect (bare propagator, no filter) ---")
print(f"  one call on |x| = 1     : {result['t_apply_unit'] * 1e3:8.1f} ms")
print(f"  one call on |x| = 1e3   : {result['t_apply_big'] * 1e3:8.1f} ms")
print(f"  ratio                   : {result['t_apply_big'] / result['t_apply_unit']:8.2f} x")

print("\n--- filtered Arnoldi ---")
print(f"  {'':24s} {'rescaled':>16s} {'plain':>16s}")
for label, key, fmt in (
    ("t per filter call [ms]", "t_filter", "16.1f"),
    ("t Arnoldi loop [s]", "t_arnoldi", "16.1f"),
    ("res_rel", "res_rel", "16.3e"),
):
    a, b = resc[key], plain[key]
    if key == "t_filter":
        a, b = a * 1e3, b * 1e3
    if key == "res_rel":
        a, b = abs(a), abs(b)
    print(f"  {label:24s} {a:{fmt}} {b:{fmt}}")

print(f"  {'speedup':24s} {plain['t_arnoldi'] / resc['t_arnoldi']:15.2f}x")

print("\n--- the answer ---")
for label, key in (("mu", "mu"), ("mu_RQ", "mu_RQ"),
                   ("rate_bf", "rate_bf"), ("rate_RQ", "rate_RQ")):
    a, b = complex(resc[key]), complex(plain[key])
    rel = abs(a - b) / abs(b)
    print(f"  {label:8s} rescaled {a.real:.12e}  plain {b.real:.12e}  rel. diff {rel:.2e}")

# The thing to judge by: is the disagreement between the two variants smaller
# than the residual each of them reports? If so, they agree to the accuracy
# either one can claim, and the rescaling is free.
d_rate = abs(complex(resc["rate_RQ"]) - complex(plain["rate_RQ"])) / abs(
    complex(plain["rate_RQ"])
)
floor = max(abs(resc["res_rel"]), abs(plain["res_rel"]))
print(
    f"\nrate_RQ disagreement {d_rate:.2e} vs worst res_rel {floor:.2e}  ->  "
    + ("INDISTINGUISHABLE" if d_rate < floor else "RESOLVED, the rescaling costs accuracy")
)

In [ ]:
import matplotlib.pyplot as plt

# Fixed hue per variant, with marker and dash carrying the same identity.
STYLE = {
    "rescaled": {"color": "#0072B2", "marker": "o", "ls": "-"},
    "plain": {"color": "#E69F00", "marker": "s", "ls": "--"},
}
T_block = result["T_block"]
m = result["settings"]["m_arnoldi"]
steps = np.arange(1, m + 1)


def rate_of(v):
    """Predicted bit-flip rate after each filtered Arnoldi step."""
    return (-np.log(np.asarray(v["mu_list"], dtype=complex)) / T_block).real


fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

# Left: both convergence histories against the plain run's converged answer.
ref = complex(plain["rate_RQ"]).real
ax = axes[0]
for key, v in (("rescaled", resc), ("plain", plain)):
    ax.plot(
        steps,
        np.abs(rate_of(v) - ref) / abs(ref),
        **STYLE[key],
        ms=3.5,
        lw=1.2,
        markevery=max(1, m // 25),
        label=f"rescale_filter={v['rescale_filter']}",
    )
ax.axhline(abs(plain["res_rel"]), color="0.35", lw=0.8, ls=":")
ax.annotate(
    "res_rel (plain)",
    xy=(0.98, abs(plain["res_rel"])),
    xycoords=("axes fraction", "data"),
    ha="right",
    va="bottom",
    fontsize=7,
    color="0.35",
)
ax.set_yscale("log")
ax.set_xlabel("filtered Arnoldi iteration")
ax.set_ylabel("relative deviation from the plain converged rate")
ax.set_title("Same answer?", fontsize=9)
ax.legend(fontsize=7, frameon=False)

# Right: the two histories against each other, step by step. This isolates the
# rescaling from convergence -- a gap here is disagreement, not immaturity.
ax = axes[1]
gap = np.abs(rate_of(resc) - rate_of(plain)) / np.abs(rate_of(plain))
ax.plot(steps, gap, color="#0072B2", marker="o", ms=3.5, lw=1.2,
        markevery=max(1, m // 25))
ax.axhline(max(abs(resc["res_rel"]), abs(plain["res_rel"])), color="0.35",
           lw=0.8, ls=":")
ax.annotate(
    "worst res_rel",
    xy=(0.98, max(abs(resc["res_rel"]), abs(plain["res_rel"]))),
    xycoords=("axes fraction", "data"),
    ha="right",
    va="bottom",
    fontsize=7,
    color="0.35",
)
ax.set_yscale("log")
ax.set_xlabel("filtered Arnoldi iteration")
ax.set_ylabel("relative gap, rescaled vs plain")
ax.set_title(
    f"Cost: {plain['t_arnoldi'] / resc['t_arnoldi']:.2f}x faster",
    fontsize=9,
)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)

fig.suptitle(
    rf"ChebAr rescale_filter, $|\alpha|^2 = {result['settings']['alpha_sq']}$",
    fontsize=10,
)
plt.show()

## How to read this

- **"the regime effect"** is the mechanism, measured without any filter: the
  same `mesolve` on a vector of norm `1e3` versus norm `1`. A ratio well above
  1 confirms the integrator is switching tolerance regimes, and bounds the
  speedup `rescale_filter` can possibly deliver.
- **Left panel / "the answer"** — the verdict is the last line of the table. If
  the two variants disagree by less than the residual either of them reports,
  they agree to the accuracy either can claim and the rescaling is free at this
  parameter point.
- **Right panel** isolates the rescaling from convergence: both curves start
  from the same `x0` and the same polynomial, so a gap that stays at the
  residual floor is noise, and a gap that grows with the iteration is the
  recurrence amplifying the looser per-call error.
- If the gap is resolved above the residual, the fix is to tighten `atol`
  rather than to set `rescale_filter=False` — the tolerance, not the rescaling,
  is what is actually being traded.
- One parameter point is not a validation. Re-run at the `alpha_sq` you care
  about before turning this on for a sweep.